# Can AI Know When It Is Wrong?
## Measuring self-reported confidence calibration with FLAN-T5

**Project:** Stanford Probability for AI · Fall 2026  
**Model:** `google/flan-t5-base` (Apache 2.0)  
**Question data:** MMLU (`cais/mmlu`, MIT license)  

This notebook measures whether a language model's *self-reported confidence* matches how often its multiple-choice answers are correct. It uses three MMLU subjects: high-school biology, high-school computer science, and high-school statistics.

**Important:** this is a small educational experiment, not a medical or other high-stakes evaluation. MMLU is a public benchmark and may have appeared in model training data; that is a limitation.

### Before running
The course asks you to write down a prediction before examining results. Review the proposed prediction below and change it if it is not genuinely yours. Do not edit it after seeing results.

In [ ]:
# PRE-REGISTER YOUR PREDICTION BEFORE RUNNING THE EXPERIMENT.
# Edit this sentence now if it does not reflect your actual belief.
PREDICTION = (
    "I am 70% confident that FLAN-T5-base will be overconfident on these "
    "questions overall, and that its calibration will differ across the three subjects."
)
print("Pre-experiment prediction:", PREDICTION)

## 1. Install the tools
Run this notebook in Google Colab with a GPU runtime if available. The model can run on CPU, but inference will be slower.

In [ ]:
!pip -q install datasets transformers sentencepiece accelerate pandas matplotlib scikit-learn

## 2. Load the question data and lock the test set
We take 100 questions from each subject, using a fixed random seed. Within each subject, 70 questions go to the development set and 30 are reserved as a **locked test set**. We inspect development results first. The test set is evaluated only in the final section.

The data split is reproducible. Do not inspect `locked_test` or run the final test cell until the development analysis is complete.

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import random, re, json, os, gc, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SUBJECTS = {
    "Biology": "high_school_biology",
    "Computer science": "high_school_computer_science",
    "Statistics": "high_school_statistics",
}

all_rows = []
for domain, subject in SUBJECTS.items():
    ds = load_dataset("cais/mmlu", subject, split="test")
    frame = ds.to_pandas()
    frame = frame.sample(n=min(100, len(frame)), random_state=SEED).reset_index(drop=True)
    frame["domain"] = domain
    frame["source_subject"] = subject
    frame["question_id"] = [f"{subject}_{i:03d}" for i in range(len(frame))]
    all_rows.append(frame)

all_data = pd.concat(all_rows, ignore_index=True)
# Shuffle within each subject, then reserve 30% as the locked test set.
dev_parts, test_parts = [], []
for domain, group in all_data.groupby("domain", sort=True):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_test = max(1, round(len(group) * 0.30))
    test_parts.append(group.iloc[:n_test].copy())
    dev_parts.append(group.iloc[n_test:].copy())

dev_data = pd.concat(dev_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
locked_test = pd.concat(test_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Development questions:", len(dev_data))
print("Locked test questions:", len(locked_test))
print("Development by subject:", dev_data.domain.value_counts())
print("Locked test is reserved. Do not inspect it yet.")

## 3. Load the model
We use FLAN-T5-base as one fixed model. This project studies one model, not all language models. The model will answer each multiple-choice question and then report a confidence percentage for its own selected answer.

In [ ]:
MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(DEVICE)
print("Loaded", MODEL_NAME, "on", DEVICE)

## 4. Ask for an answer and a confidence estimate
The answer and confidence are requested in separate prompts. Confidence is the model's own stated estimate, not a probability extracted from its token logits. This distinction matters when interpreting the results.

In [ ]:
def generate_text(prompt, max_new_tokens=20):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

LETTERS = ["A", "B", "C", "D"]

def ask_model(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{letter}. {choice}" for letter, choice in zip(LETTERS, choices))
    question_prompt = (
        "Answer this multiple-choice question. Choose exactly one letter: A, B, C, or D.\n"
        f"Question: {row['question']}\n{choices_text}\nAnswer letter:"
    )
    raw_answer = generate_text(question_prompt, max_new_tokens=8)
    match = re.search(r"\b([A-D])\b", raw_answer.upper())
    answer_letter = match.group(1) if match else None

    if answer_letter is None:
        retry = generate_text("Return only one letter A, B, C, or D. No explanation. " + question_prompt, max_new_tokens=4)
        match = re.search(r"\b([A-D])\b", retry.upper())
        answer_letter = match.group(1) if match else None
        raw_answer += " | retry: " + retry

    if answer_letter is None:
        raw_confidence = "unavailable"
        confidence = None
    else:
        confidence_prompt = (
            "You selected answer " + answer_letter + " for this multiple-choice question. "
            "How confident are you that this selected answer is correct? "
            "Return only one whole-number percentage from 0 to 100. Do not explain.\n"
            f"Question: {row['question']}\n{choices_text}\nYour answer: {answer_letter}\nConfidence percentage:"
        )
        raw_confidence = generate_text(confidence_prompt, max_new_tokens=8)
        nums = re.findall(r"\b(100|[1-9]?\d)\b", raw_confidence)
        confidence = int(nums[0]) if nums else None

    correct_index = int(row["answer"])
    correct_letter = LETTERS[correct_index]
    return {
        "question_id": row["question_id"],
        "domain": row["domain"],
        "question": row["question"],
        "model_answer": answer_letter,
        "correct_answer": correct_letter,
        "correct": int(answer_letter == correct_letter) if answer_letter else None,
        "confidence": confidence,
        "raw_answer": raw_answer,
        "raw_confidence": raw_confidence,
    }

## 5. Run the development set first
This may take several minutes. The notebook saves the raw development predictions so the analysis can be repeated without asking the model again.

In [ ]:
from tqdm.auto import tqdm

def run_split(frame, output_path):
    if os.path.exists(output_path):
        print("Found existing file:", output_path, "— loading it. Delete it only if you intentionally want to rerun inference.")
        return pd.read_csv(output_path)
    records = []
    for _, row in tqdm(frame.iterrows(), total=len(frame)):
        records.append(ask_model(row))
    result = pd.DataFrame(records)
    result.to_csv(output_path, index=False)
    return result

dev_results = run_split(dev_data, "development_results.csv")
print("Development inference complete.")
print("Unparsed answers:", int(dev_results.model_answer.isna().sum()))
print("Unparsed confidence values:", int(dev_results.confidence.isna().sum()))

## 6. Development analysis
A confidence bucket groups answers with similar stated confidence. For each bucket, compare the average confidence with the fraction actually correct. The **gap** is the absolute difference between those two percentages. The overall **average gap** is the bucket gaps averaged with each bucket weighted by its number of answers.

- If observed accuracy is lower than stated confidence, the model is **overconfident**.
- If observed accuracy is higher than stated confidence, the model is **underconfident**.
- If they are close, it is better calibrated in that bucket.

In [ ]:
import matplotlib.pyplot as plt

BUCKETS = [0, 20, 40, 60, 80, 101]
BUCKET_LABELS = ["0–19", "20–39", "40–59", "60–79", "80–100"]

def calibration_table(results):
    valid = results.dropna(subset=["confidence", "correct"]).copy()
    valid["bucket"] = pd.cut(valid.confidence, bins=BUCKETS, labels=BUCKET_LABELS, right=False, include_lowest=True)
    rows = []
    for label in BUCKET_LABELS:
        part = valid[valid.bucket == label]
        if len(part):
            mean_conf = part.confidence.mean() / 100
            accuracy = part.correct.mean()
            rows.append({"bucket": label, "n": len(part), "mean_confidence": mean_conf,
                         "accuracy": accuracy, "gap": abs(mean_conf - accuracy),
                         "direction": "overconfident" if mean_conf > accuracy else "underconfident" if mean_conf < accuracy else "calibrated"})
    return pd.DataFrame(rows)

def average_gap(table):
    if table.empty:
        return float("nan")
    return float((table["gap"] * table["n"]).sum() / table["n"].sum())

dev_table = calibration_table(dev_results)
dev_valid = dev_results.dropna(subset=["confidence", "correct"])
print("Development accuracy:", round(dev_valid.correct.mean() * 100, 1), "%")
print("Development average gap:", round(average_gap(dev_table) * 100, 1), "percentage points")
display(dev_table)

In [ ]:
# Reliability diagram for the development set
if not dev_table.empty:
    plt.figure(figsize=(7, 6))
    plt.plot([0, 1], [0, 1], "--", label="Perfect calibration")
    plt.plot(dev_table.mean_confidence, dev_table.accuracy, "o-", linewidth=2, label="FLAN-T5-base")
    for _, r in dev_table.iterrows():
        plt.annotate(f"n={int(r.n)}", (r.mean_confidence, r.accuracy), xytext=(5, 5), textcoords="offset points")
    plt.xlabel("Mean self-reported confidence")
    plt.ylabel("Observed fraction correct")
    plt.title("Development set: confidence vs. accuracy")
    plt.xlim(0, 1); plt.ylim(0, 1); plt.grid(alpha=.25); plt.legend(); plt.tight_layout()
    plt.savefig("development_reliability.png", dpi=180)
    plt.show()

## 7. Compare the three subjects (development only)

In [ ]:
subject_rows = []
for domain, part in dev_results.groupby("domain"):
    valid = part.dropna(subset=["confidence", "correct"])
    table = calibration_table(valid)
    subject_rows.append({"domain": domain, "n_valid": len(valid), "accuracy": valid.correct.mean() if len(valid) else np.nan,
                         "average_gap": average_gap(table)})
subject_summary = pd.DataFrame(subject_rows)
display(subject_summary)

## 8. Open the locked test set — only now
Do not run this section until you have finished interpreting the development set and written down what you expect on the test set. This cell runs the model on the reserved questions for the first and only planned evaluation.

In [ ]:
# FINAL TEST: run once, after development analysis is complete.
test_results = run_split(locked_test, "locked_test_results.csv")
test_table = calibration_table(test_results)
test_valid = test_results.dropna(subset=["confidence", "correct"])
print("LOCKED TEST accuracy:", round(test_valid.correct.mean() * 100, 1), "%")
print("LOCKED TEST average gap:", round(average_gap(test_table) * 100, 1), "percentage points")
display(test_table)

In [ ]:
# Final test reliability diagram
if not test_table.empty:
    plt.figure(figsize=(7, 6))
    plt.plot([0, 1], [0, 1], "--", label="Perfect calibration")
    plt.plot(test_table.mean_confidence, test_table.accuracy, "o-", linewidth=2, label="Locked test")
    for _, r in test_table.iterrows():
        plt.annotate(f"n={int(r.n)}", (r.mean_confidence, r.accuracy), xytext=(5, 5), textcoords="offset points")
    plt.xlabel("Mean self-reported confidence")
    plt.ylabel("Observed fraction correct")
    plt.title("Locked test set: confidence vs. accuracy")
    plt.xlim(0, 1); plt.ylim(0, 1); plt.grid(alpha=.25); plt.legend(); plt.tight_layout()
    plt.savefig("locked_test_reliability.png", dpi=180)
    plt.show()

In [ ]:
# Export only actual measured summaries for the final report/dashboard.
summary = {
    "project_title": "Can AI Know When It Is Wrong?",
    "model": MODEL_NAME,
    "dataset": "cais/mmlu",
    "prediction": PREDICTION,
    "development": {
        "n_questions": int(len(dev_results)),
        "n_valid": int(len(dev_valid)),
        "accuracy": float(dev_valid.correct.mean()) if len(dev_valid) else None,
        "average_gap": average_gap(dev_table),
        "buckets": dev_table.to_dict(orient="records"),
    },
    "locked_test": {
        "n_questions": int(len(test_results)),
        "n_valid": int(len(test_valid)),
        "accuracy": float(test_valid.correct.mean()) if len(test_valid) else None,
        "average_gap": average_gap(test_table),
        "buckets": test_table.to_dict(orient="records"),
    },
    "subject_development": subject_summary.replace({np.nan: None}).to_dict(orient="records"),
    "note": "Results are from a single model and a small sample of MMLU questions. Public benchmark contamination is possible. Self-reported confidence is not a calibrated probability by definition."
}
with open("project_results.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved project_results.json")

## 9. What to report honestly
Report the development and locked-test results separately. Include the number of questions with parseable answers and confidence estimates. Do not hide failed parses. Discuss that (1) the sample is small, (2) MMLU is public and may have been seen during pretraining, (3) self-reported confidence can be affected by the prompt, and (4) results apply only to this model, prompt, and sample.

**No result is pre-filled in this notebook.** The final writeup must use the numbers produced by your run.